# Consume data from Event Hub using Spark Structured Streaming

In [0]:
SCOPE = dbutils.widgets.get("kv_scope")
EVENT_HUB_NAMESPACE = dbutils.secrets.get(scope=SCOPE, key="oles0305-evh-namespace")
EVENT_HUB_NAME = dbutils.secrets.get(scope=SCOPE, key="oles0305-evh-name")
CONNECTION_STRING = dbutils.secrets.get(scope=SCOPE, key="oles0305-evh-connection-str")

KEY_EVH_NAME = dbutils.widgets.get("kv_evh_name")
KEY_EVH_NAMESPACE = dbutils.widgets.get("kv_evh_namespace")
KEY_CONN_STR = dbutils.widgets.get("kv_evh_connection_str")

EVENT_HUB_NAME = dbutils.secrets.get(scope=SCOPE, key=KEY_EVH_NAME)
EVENT_HUB_NAMESPACE = dbutils.secrets.get(scope=SCOPE, key=KEY_EVH_NAMESPACE)
CONNECTION_STRING = dbutils.secrets.get(scope=SCOPE, key=KEY_CONN_STR)

CHECKPOINT_LOCATION = dbutils.widgets.get("checkpoint_location")
TARGET_TABLE = dbutils.widgets.get("target_table")

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": f"{EVENT_HUB_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": EVENT_HUB_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password=\"{CONNECTION_STRING}\";',
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "30000",
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

In [0]:
df_raw = (
    spark.readStream
        .format("kafka")
        .options(**kafka_options)
        .load()
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

df_bronze = (
    df_raw
        .select(
            col("value").cast("string").alias("raw_json"),
            col("topic").alias("evh_name"),
            col("partition").alias("evh_partition"),
            col("offset").alias("evh_offset"),
            col("timestamp").alias("evh_timestamp"),
            current_timestamp().alias("evh_processed_timestamp")
        )
)

In [0]:
write_query = (
    df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_LOCATION)
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
)

write_query.awaitTermination()